# W7D5 — Two Recommenders, One Dataset, Honest Metrics — Guided

**Week 7 · Day 5 · Retrieval, RAG and Recommenders** · Lab

The last taught lab of the bootcamp, and it ends where the first one started: **with a baseline.**

You build two recommenders on the same ratings — one from who rated what, one from what the films
are about — evaluate both on a chronological split with four ranking metrics, and put a third
column beside them: recommend the most-rated films to everybody. That column is not a joke. It
wins today, and knowing that it wins is the difference between a recommender you can defend and
one you can demo.

Then the cold start, demonstrated rather than described: 1,800-odd films that no collaborative
method can ever recommend, because nobody has rated them yet.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٧ اليوم ٥ — نظاما توصية، بيانات واحدة، مقاييس أمينة

**الأسبوع السابع · اليوم الخامس · الاسترجاع والتوليد المعزّز والتوصية** · معمل

آخر معمل مُدرَّس في المعسكر، وينتهي من حيث بدأ أوّله: **من خطّ الأساس.**

تبني نظامَي توصية على التقييمات نفسها — أحدهما ممّن قيّم ماذا، والآخر ممّا تدور عنه الأفلام —
وتقيّمهما على تقسيم زمني بأربعة مقاييس ترتيب، وتضع بجوارهما عمودًا ثالثًا: رشّح للجميع أكثر الأفلام
تقييمًا. وهذا العمود ليس مزحة. إنه يفوز اليوم، ومعرفتك أنه يفوز هي الفرق بين نظام توصية تدافع عنه
ونظام تعرضه فقط.

ثم البداية الباردة، معروضةً لا موصوفة: نحو ألف وثمانمئة فيلم لا تستطيع أي طريقة تعاونية أن ترشّحها
أبدًا، لأن أحدًا لم يقيّمها بعد.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Build an item-item similarity over co-rating users only, and handle the undefined case explicitly.
- Say why a similarity of 1.0000 can be worthless, and count how many of yours are that kind.
- Build a content-based recommender from text, and say what it can do that collaborative cannot.
- Split ratings chronologically and say what a random split would have leaked.
- Compute precision@k, recall@k, NDCG@k and MAP, and say which one hides what.
- Put a popularity baseline beside every model you built, and read the result honestly.
- Demonstrate both halves of cold start and measure a switching hybrid against both models.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تبني تشابهًا بين العناصر على المستخدمين المشتركين في التقييم فقط، وأن تعالج الحالة غير
  المعرَّفة صراحةً.
- أن تقول لماذا قد يكون تشابه ١٫٠٠٠٠ بلا قيمة، وأن تعدّ كم من تشابهاتك من هذا النوع.
- أن تبني نظام توصية قائمًا على المحتوى من النصّ، وأن تقول ما يستطيعه ولا يستطيعه التعاوني.
- أن تقسّم التقييمات زمنيًّا وأن تقول ماذا كان التقسيم العشوائي ليُسرّب.
- أن تحسب الدقّة عند k والاستدعاء عند k وNDCG وMAP، وأن تقول ماذا يخفي كلٌّ منها.
- أن تضع خطّ أساس شعبيّ بجوار كل نموذج بنيته، وأن تقرأ النتيجة بأمانة.
- أن تعرض نصفَي البداية الباردة وأن تقيس هجينًا مبدِّلًا مقابل النموذجين.

</div>

## About the data

**`movies_ratings`** — MovieLens `ml-latest-small`: **100,836 ratings** by **610 users** over
**9,724 rated films**, each row carrying the rating, its timestamp, and the film's title and
genres. GroupLens' own data, under the MovieLens terms for research and teaching.

One row is one person rating one film at one moment. Title and genre ride along on every row on
purpose: the collaborative recommender and the content-based one must be built from **the same
file**, or the comparison in task 4 is between two datasets rather than two methods.

**The known problem, and it is the first thing you measure:** the user-item matrix is about
**98% empty**. Every method here is an attempt to say something about the 98% from the 2%, and any
evaluation that treats a missing rating as "disliked" is measuring the sampling, not the taste.

**Two further properties you will run into, and both are honest:**

- **No user has fewer than 20 ratings.** MovieLens filtered them out before publishing. So the
  thin-history half of cold start cannot be found in the data — task 5 *constructs* it by hiding
  ratings, and says so.
- **1,857 films are rated only after the split point.** That half of cold start is real and needs
  no construction: those films exist in the catalogue and have no ratings to learn from.

<div dir="rtl" align="right">

## عن البيانات

**`movies_ratings`** — مجموعة MovieLens الصغيرة: **١٠٠٬٨٣٦ تقييمًا** من **٦١٠ مستخدمين** على
**٩٬٧٢٤ فيلمًا مُقيَّمًا**، وفي كل صفّ التقييم وطابعه الزمني وعنوان الفيلم وأنواعه. وهي بيانات
GroupLens نفسها، بشروط MovieLens للبحث والتعليم.

والصفّ الواحد شخصٌ يقيّم فيلمًا في لحظة. والعنوان والنوع في كل صفّ عن قصد: فيجب أن يُبنى النظام
التعاوني والنظام القائم على المحتوى من **الملف نفسه**، وإلا صارت المقارنة في المهمة الرابعة بين
بياناتين لا بين طريقتين.

**والمشكلة المعروفة، وهي أول ما تقيسه:** مصفوفة المستخدم-العنصر فارغة بنحو **٩٨٪**. وكل طريقة هنا
محاولةٌ لقول شيء عن الـ٩٨٪ من الـ٢٪، وأي تقييم يعامل التقييم المفقود على أنه «غير معجب» يقيس أخذ
العيّنة لا الذوق.

**وصفتان أخريان ستقابلهما، وكلتاهما أمينة:**

- **لا مستخدم بأقلّ من عشرين تقييمًا**، فقد رشّحتهم MovieLens قبل النشر. فنصف البداية الباردة
  الخاصّ بالسجلّ الضئيل غير موجود في البيانات — والمهمة الخامسة **تصنعه** بإخفاء تقييمات، وتقول ذلك.
- **و١٬٨٥٧ فيلمًا لا تُقيَّم إلا بعد نقطة التقسيم.** وهذا النصف حقيقي ولا يحتاج صناعة: أفلام في
  الفهرس بلا تقييمات يُتعلَّم منها.

</div>

## Setup

Two constants decide how much of the data the collaborative half sees, and both are choices you
should be able to defend:

- `MIN_ITEM_RATINGS = 20` — films with fewer ratings are dropped from the item-item matrix. Without
  this, most similarities rest on one or two users, which is exactly the failure the warm-up shows.
- `HOLDOUT = 0.2` — each user's most recent 20% of ratings is the test set.

<div dir="rtl" align="right">

## الإعداد

ثابتان يقرّران كم من البيانات يرى النصف التعاوني، وكلاهما اختيار ينبغي أن تستطيع الدفاع عنه:

- `MIN_ITEM_RATINGS = 20` — تُستبعد الأفلام الأقلّ تقييمًا من مصفوفة العناصر. وبدونه تقوم أكثر
  التشابهات على مستخدم أو مستخدمين، وهو بعينه الإخفاق الذي يُظهره الإحماء.
- `HOLDOUT = 0.2` — أحدث ٢٠٪ من تقييمات كل مستخدم هي مجموعة الاختبار.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, report
from aiep.viz import use_course_style, savefig

ensure("sentence-transformers", "matplotlib", "pandas", "pyarrow")
seed_everything(42)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

use_course_style()
np.set_printoptions(precision=4, suppress=True)

EMBEDDER = "sentence-transformers/all-MiniLM-L6-v2"
MIN_ITEM_RATINGS = 20      # films below this leave the item-item matrix
MIN_CO_RATERS = 3          # a similarity from fewer than this is not computed
HOLDOUT = 0.2              # the most recent fifth of each user's ratings
LIKED = 4.0                # a test rating at or above this counts as relevant
TOP_K, PRECISION_K = 10, 3

RATINGS = pd.read_parquet(get_dataset("movies_ratings")).sort_values(["userId", "timestamp"])
CATALOGUE = RATINGS.drop_duplicates("movieId").set_index("movieId")[["title", "genres", "year"]]

print(f"{len(RATINGS):,} ratings · {RATINGS.userId.nunique()} users · "
      f"{RATINGS.movieId.nunique():,} films")
print(f"ratings per user: min {RATINGS.groupby('userId').size().min()}, "
      f"median {RATINGS.groupby('userId').size().median():.0f}")
print(versions(), "| device:", device())

## Section 1 — Warm-up: four numbers, and two of them are traps  (≈25 min)

This morning's 4×5 matrix, blanks left blank:

```
        A    B    C    D    E
U1      5    3    —    4    —
U2      4    3    —    —    2
U3      —    —    5    4    —
U4      5    —    —    5    —
```

Four results, and you should be able to check every one by hand:

1. **`sim(A, D) = 0.9939`** — over the **two** users who rated both.
2. **`sim(A, C)` is undefined.** Nobody rated both. It is not zero, and the code must say `None`
   rather than pretend. Treating it as zero drags every prediction towards the mean, silently.
3. **`sim(B, D) = 1.0000` exactly** — from **one** shared user. In one dimension, any two positive
   numbers are perfectly similar. A perfect score computed from a single rating is evidence of
   nothing at all, and it will still be the top of your neighbour list.
4. **Predicted rating for U2 on D: 3.4985**, i.e. **3.50** — and look at what carried it there: the
   1.0000 from trap 3 is the heaviest weight in the average.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: أربعة أرقام، اثنان منها فخّان (نحو ٢٥ دقيقة)

مصفوفة الصباح ٤×٥، والفراغات فراغات.

أربع نتائج، وتستطيع أن تتحقّق من كل واحدة بيدك:

١. **`sim(A, D) = 0.9939`** — على **المستخدمَين** اللذين قيّما الاثنين.
٢. **`sim(A, C)` غير معرَّف.** لم يقيّم أحد الاثنين. وليس صفرًا، وعلى الرمز أن يقول `None` لا أن
   يتظاهر. فمعاملته صفرًا تجرّ كل تنبّؤ نحو المتوسّط، بصمت.
٣. **`sim(B, D) = 1.0000` تمامًا** — من **مستخدم واحد** مشترك. ففي بُعد واحد يكون أي عددين موجبين
   متشابهَين تمامًا. والدرجة التامّة المحسوبة من تقييم واحد ليست دليلًا على شيء البتّة، وستظلّ مع
   ذلك في رأس قائمة جيرانك.
٤. **التقييم المتنبّأ به لـU2 على D هو 3.4985**، أي **3.50** — وانظر ما الذي حمله إلى هناك: الـ
   1.0000 من الفخّ الثالث هي أثقل وزن في المتوسّط.

</div>

In [ ]:
WARMUP = {"U1": {"A": 5, "B": 3, "D": 4},
          "U2": {"A": 4, "B": 3, "E": 2},
          "U3": {"C": 5, "D": 4},
          "U4": {"A": 5, "D": 5}}


def column(item):
    """Everyone who rated this item, and what they gave it."""
    return {user: row[item] for user, row in WARMUP.items() if item in row}


def similarity(first, second):
    """Cosine between two item columns over the users who rated BOTH.

    Returns (value, co_raters). The value is None when nobody rated both — which is a
    different thing from zero, and the difference is the whole of this warm-up.
    """
    left, right = column(first), column(second)
    shared = sorted(set(left) & set(right))
    if not shared:
        return None, 0
    a = np.array([left[user] for user in shared], dtype=float)
    b = np.array([right[user] for user in shared], dtype=float)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))), len(shared)


for pair in [("A", "D"), ("A", "C"), ("B", "D")]:
    value, co_raters = similarity(*pair)
    shown = "undefined" if value is None else f"{value:.4f}"
    print(f"sim{pair} = {shown:>9}   from {co_raters} co-rating user(s)")

# U2 has not rated D. Of U2's rated items, A and D and B and D both have a similarity.
NEIGHBOURS = [("A", WARMUP["U2"]["A"], similarity("A", "D")[0]),
              ("B", WARMUP["U2"]["B"], similarity("B", "D")[0])]
PREDICTION = (sum(sim * rating for _, rating, sim in NEIGHBOURS)
              / sum(sim for _, _, sim in NEIGHBOURS))

print(f"\npredicting U2's rating for D from {[n for n, _, _ in NEIGHBOURS]}:")
for name, rating, sim in NEIGHBOURS:
    print(f"  {name}: rated {rating} × similarity {sim:.4f}")
print(f"  = {PREDICTION:.4f} → {PREDICTION:.2f}")
print("  the 1.0000 is the heavier of the two weights, and it came from one person")

WARM_SIM_AD = similarity("A", "D")[0]
WARM_SIM_AC = similarity("A", "C")[0]
WARM_SIM_BD, WARM_CO_BD = similarity("B", "D")

### The ranking metrics, on the slide's three orderings

One user, a list of five recommendations, three of them relevant. Three orderings:

| Relevant at | precision@3 | NDCG@10 |
|---|---|---|
| 1, 2, 5 | **0.667** | **0.9469** |
| 1, 4, 5 | **0.333** | **0.8529** |
| 1, 2, 4 | **0.667** | **0.9675** |

Rows one and three are the pair to stare at. Moving a relevant item from position 5 to position 4
**leaves precision@3 exactly where it was** and moves NDCG. Precision@k cannot see anything outside
its cut; NDCG can see the whole list and cares where in it things landed.

Neither is "the better metric". They answer different questions, and task 6 asks you which question
your interface is actually asking.

<div dir="rtl" align="right">

### مقاييس الترتيب، على ترتيبات الشريحة الثلاثة

مستخدم واحد، وقائمة من خمس توصيات، ثلاث منها ذات صلة. وثلاثة ترتيبات في الجدول أعلاه.

والصفّان الأول والثالث هما ما ينبغي التحديق فيه. فنقل عنصر ذي صلة من الموضع الخامس إلى الرابع
**يترك الدقّة عند ٣ حيث كانت تمامًا** ويحرّك NDCG. فالدقّة عند k لا ترى شيئًا خارج قطعها، وNDCG ترى
القائمة كلها وتُعنى بأين وقع كل شيء فيها.

ولا أحدهما «المقياس الأفضل». إنهما يجيبان عن سؤالين مختلفين، والمهمة السادسة تسألك أي سؤال تطرحه
واجهتك أنت فعلًا.

</div>

In [ ]:
LIST_LENGTH, RELEVANT_COUNT = 5, 3


def precision_at(positions, k=PRECISION_K):
    """Fraction of the first k slots holding something relevant."""
    return sum(1 for position in positions if position <= k) / k


def ndcg_of(positions, length=LIST_LENGTH, relevant=RELEVANT_COUNT):
    """Discounted gain over the whole list, divided by the best possible arrangement."""
    gains = [1.0 if (i + 1) in positions else 0.0 for i in range(length)]
    dcg = sum(gain / np.log2(i + 2) for i, gain in enumerate(gains))
    ideal = sum(1 / np.log2(i + 2) for i in range(relevant))
    return dcg / ideal


ORDERINGS = {"1, 2, 5": (1, 2, 5), "1, 4, 5": (1, 4, 5), "1, 2, 4": (1, 2, 4)}
WARM_METRICS = {name: (precision_at(positions), ndcg_of(positions))
                for name, positions in ORDERINGS.items()}

for name, (precision, ndcg) in WARM_METRICS.items():
    print(f"relevant at {name}:  precision@3 {precision:.3f}   NDCG {ndcg:.4f}")

print(f"\nthe discounts themselves: "
      f"{[round(1 / np.log2(i + 2), 4) for i in range(LIST_LENGTH)]}")
print("moving one item from position 5 to 4: precision@3 "
      f"{WARM_METRICS['1, 2, 5'][0]:.3f} → {WARM_METRICS['1, 2, 4'][0]:.3f} (unchanged), "
      f"NDCG {WARM_METRICS['1, 2, 5'][1]:.4f} → {WARM_METRICS['1, 2, 4'][1]:.4f}")

## Section 2 — Core: six tasks  (≈60 min)

1. Build the rating matrix and measure the sparsity you were warned about.
2. Item-based collaborative filtering, with the undefined case handled and the co-rater counts kept.
3. Content-based recommendations from title and genre text.
4. Evaluate both on a chronological split — **with a popularity baseline beside them**.
5. Cold start, both halves, and a switching hybrid measured against both models.
6. The metrics task: which number would you put on a phone screen, and why.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. ابنِ مصفوفة التقييمات وقِس التناثر الذي حُذّرت منه.
٢. الترشيح التعاوني القائم على العناصر، مع معالجة الحالة غير المعرَّفة وحفظ أعداد المقيّمين المشتركين.
٣. التوصية القائمة على المحتوى من نصّ العنوان والنوع.
٤. قيّم الاثنين على تقسيم زمني — **وبجوارهما خطّ الأساس الشعبي**.
٥. البداية الباردة بنصفيها، وهجينٌ مبدِّل يُقاس مقابل النموذجين.
٦. مهمة المقاييس: أي رقم تضعه على شاشة هاتف، ولماذا.

</div>

### Task 2.1 — the matrix, and the number that governs everything after it

Split each user's ratings chronologically first — the most recent `HOLDOUT` of **their own**
ratings is their test set — then build the user-item matrix from the training half.

Report the sparsity of the **full** matrix, before any filtering: 610 users × 9,724 films against
100,836 ratings. Then apply `MIN_ITEM_RATINGS` and report the sparsity again. Both numbers are
true and they are not the same number, which is why a paper that reports one without saying which
is not telling you much.

Then write, in the markdown cell below, **what treating a missing rating as "disliked" would do**
to every metric in task 4.

<div dir="rtl" align="right">

### المهمة ٢٫١ — المصفوفة، والرقم الذي يحكم كل ما بعده

قسّم تقييمات كل مستخدم زمنيًّا أولًا — فأحدث `HOLDOUT` من تقييماته **هو** هو مجموعة اختباره — ثم
ابنِ مصفوفة المستخدم-العنصر من نصف التدريب.

واذكر تناثر المصفوفة **الكاملة** قبل أي ترشيح: ٦١٠ مستخدمين × ٩٬٧٢٤ فيلمًا مقابل ١٠٠٬٨٣٦ تقييمًا.
ثم طبّق `MIN_ITEM_RATINGS` واذكر التناثر ثانيةً. والرقمان صحيحان وليسا الرقم نفسه، ولهذا فالورقة
التي تذكر أحدهما بلا بيان أيّهما لا تقول لك كثيرًا.

ثم اكتب في خلية markdown أدناه **ماذا يفعل اعتبار التقييم المفقود «غير معجب»** بكل مقياس في المهمة
الرابعة.

</div>

In [ ]:

# TODO: Mark the most recent HOLDOUT of each user's ratings as the test set.
# مهمة: علّم أحدث نسبة `HOLDOUT` من تقييمات كل مستخدم بأنها مجموعة الاختبار.

TRAIN = RATINGS[~RATINGS.is_test]
TEST = RATINGS[RATINGS.is_test]

# TODO: Compute the sparsity of the full user-item matrix, then of the filtered one.
# مهمة: احسب تناثر مصفوفة المستخدم-العنصر الكاملة، ثم المرشَّحة.

print(f"train {len(TRAIN):,} ratings · test {len(TEST):,} ratings")
print(f"full matrix   {RATINGS.userId.nunique()} × {RATINGS.movieId.nunique():,} "
      f"→ {FULL_SPARSITY:.1%} empty")
print(f"filtered      {MATRIX.shape[0]} × {MATRIX.shape[1]:,} "
      f"→ {CORE_SPARSITY:.1%} empty  (films with ≥ {MIN_ITEM_RATINGS} ratings)")
print(f"\nfilms dropped by the filter: "
      f"{TRAIN.movieId.nunique() - len(KEPT_ITEMS):,} of {TRAIN.movieId.nunique():,} — "
      f"and they are {1 - CORE.shape[0] / TRAIN.shape[0]:.1%} of the ratings")

**Your answer.** What would treating every missing rating as "disliked" do to precision, recall and
NDCG in task 4? Replace this text with two or three sentences.

> …

<div dir="rtl" align="right">

**جوابك.** ماذا يفعل اعتبار كل تقييم مفقود «غير معجب» بالدقّة والاستدعاء وNDCG في المهمة الرابعة؟
استبدل هذا النصّ بجملتين أو ثلاث.

> …

</div>

### Task 2.2 — item-item similarity, over co-raters only

Compute the cosine between every pair of item columns **over the users who rated both**, exactly as
in the warm-up but for a thousand films at once.

Two things must survive the scale-up, and they are what the task is for:

1. **An undefined similarity stays undefined.** Pairs with no co-raters must be `NaN`, not `0.0`.
2. **Every similarity remembers how many users it came from.** Keep the co-rater count matrix, and
   report how many pairs rest on a single user. That count is the warm-up's `1.0000` trap, at scale.

Then require `MIN_CO_RATERS` before a similarity is allowed to influence anything, and say in the
output how many pairs that removed.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — تشابه العناصر، على المشتركين في التقييم فقط

احسب جيب التمام بين كل زوج من أعمدة العناصر **على المستخدمين الذين قيّموا الاثنين**، تمامًا كما في
الإحماء لكن لألف فيلم دفعةً واحدة.

وشيئان يجب أن ينجوا من التوسّع، وهما مقصود المهمة:

١. **التشابه غير المعرَّف يبقى غير معرَّف.** فالأزواج بلا مقيّم مشترك تكون `NaN` لا `0.0`.
٢. **وكل تشابه يتذكّر من كم مستخدمًا جاء.** فاحتفظ بمصفوفة أعداد المشتركين، واذكر كم زوجًا يقوم على
   مستخدم واحد. وذلك العدد هو فخّ الـ`1.0000` في الإحماء، على مقياس كبير.

ثم اشترط `MIN_CO_RATERS` قبل أن يُسمح لتشابه بالتأثير في شيء، وقل في المخرجات كم زوجًا حذف ذلك.

</div>

In [ ]:

# TODO: Build the co-rated cosine similarity matrix, the co-rater count matrix, and keep undefined pairs as NaN rather than zero.
# مهمة: ابنِ مصفوفة تشابه جيب التمام على المشتركين، ومصفوفة أعداد المشتركين، وأبقِ الأزواج غير المعرَّفة `NaN` لا صفرًا.

pairs = SIMILARITY.size - len(SIMILARITY)
print(f"{pairs:,} ordered item pairs")
print(f"  undefined (no co-raters):        {int(np.isnan(SIMILARITY).sum() - len(SIMILARITY)):,}")
print(f"  resting on exactly one user:     {LONELY_PAIRS:,} "
      f"— every one of these can score 1.0000")
print(f"  surviving MIN_CO_RATERS = {MIN_CO_RATERS}:      "
      f"{int(np.isfinite(TRUSTED).sum()):,}")

lonely_values = SIMILARITY[(CO_RATERS == 1) & np.isfinite(SIMILARITY)]
print(f"\nsimilarities computed from one user: mean {lonely_values.mean():.4f}, "
      f"{(lonely_values > 0.99).mean():.1%} of them above 0.99")
print("That is the warm-up's trap, one thousand films wide.")

SIMILARITY_USED = np.nan_to_num(TRUSTED)   # zeros only after the count is on record

### Task 2.3 — content-based, from the text

Nothing in this recommender knows who rated what. Embed each film's title and genres with
`all-MiniLM-L6-v2` — week 6's tool, unchanged — build a **taste profile** for each user as the mean
embedding of the films they rated at or above `LIKED`, and score every unseen film by cosine
against that profile.

This is a different kind of system, not a better one. It can recommend a film released this morning
that nobody has rated, and it cannot tell you that people who liked *this* also liked something with
no words in common.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — القائم على المحتوى، من النصّ

لا شيء في نظام التوصية هذا يعرف من قيّم ماذا. ضمّن عنوان كل فيلم وأنواعه بـ`all-MiniLM-L6-v2` —
أداة الأسبوع السادس بلا تغيير — وابنِ **ملفّ ذوق** لكل مستخدم بمتوسّط تضمينات الأفلام التي قيّمها
بـ`LIKED` فأعلى، واحسب لكل فيلم لم يره جيبَ التمام مقابل ذلك الملف.

وهذا نوع مختلف من الأنظمة لا نوع أفضل. فهو يستطيع أن يرشّح فيلمًا صدر هذا الصباح ولم يقيّمه أحد،
ولا يستطيع أن يقول لك إن من أحبّ *هذا* أحبّ شيئًا لا تشترك معه في كلمة.

</div>

In [ ]:

model = SentenceTransformer(EMBEDDER)

# TODO: Embed one text per film — title plus genres — normalised.
# مهمة: ضمّن نصًّا واحدًا لكل فيلم — العنوان مع الأنواع — مع التطبيع.


# TODO: Score every film for a user from the mean embedding of the films they liked.
# مهمة: احسب درجة كل فيلم لمستخدم من متوسّط تضمينات الأفلام التي أحبّها.


def collaborative_scores(row):
    """Sum of similarity × rating over the films this user has rated."""
    rated = row > 0
    scores = SIMILARITY_USED[rated].T @ row[rated]
    scores[rated] = -np.inf
    return scores


POPULARITY = RATED.sum(axis=0)


def popularity_scores(row):
    """The baseline: recommend what most people rated, to everybody."""
    scores = POPULARITY.astype(float).copy()
    scores[row > 0] = -np.inf
    return scores


example_user = USERS[0]
example_row = MATRIX[USER_INDEX[example_user]]
for label, scorer in [("collaborative", collaborative_scores),
                      ("content", content_scores),
                      ("popularity", popularity_scores)]:
    top = np.argsort(-scorer(example_row))[:3]
    titles = [CATALOGUE.title[KEPT_ITEMS[i]] for i in top]
    print(f"user {example_user} · {label:<14} {titles}")

### Task 2.4 — evaluate all three, and read the result honestly

The split is **chronological, per user**: everything you train on happened before everything you
are scored on, for that user. A random split would put a rating from 2018 in the training set and a
rating from 2016 in the test set, and your model would be predicting the past from the future — the
same leakage W2D5 checked for, wearing a different hat.

For each evaluable user, take the top 10 recommendations and score four metrics against the films
they rated at or above `LIKED` in their test period:

- **precision@3** — of the three slots you would actually show, how many were relevant?
- **recall@10** — of everything they liked, how much did you find?
- **NDCG@10** — the whole list, discounted by position.
- **MAP@10** — the average of the precisions at each hit, which rewards early hits twice.

Then read the table. The popularity baseline is likely to win, and if it does, that is the finding —
not a bug to fix before the deadline.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — قيّم الثلاثة، واقرأ النتيجة بأمانة

التقسيم **زمني لكل مستخدم**: فكل ما تدرّب عليه وقع قبل كل ما تُقاس عليه، لذلك المستخدم. والتقسيم
العشوائي يضع تقييمًا من ٢٠١٨ في التدريب وتقييمًا من ٢٠١٦ في الاختبار، فيصير نموذجك يتنبّأ بالماضي من
المستقبل — وهو التسريب نفسه الذي فحصه الأسبوع الثاني اليوم الخامس بثوب آخر.

ولكل مستخدم قابل للتقييم، خذ أفضل عشر توصيات وقِس أربعة مقاييس مقابل الأفلام التي قيّمها بـ`LIKED`
فأعلى في فترة اختباره.

ثم اقرأ الجدول. والأرجح أن يفوز خطّ الأساس الشعبي، وإن فاز فتلك هي النتيجة — لا خلل تُصلحه قبل
الموعد.

</div>

In [ ]:

RELEVANT = (TEST[(TEST.rating >= LIKED) & (TEST.movieId.isin(set(KEPT_ITEMS)))]
            .groupby("userId").movieId.apply(set).to_dict())
EVAL_USERS = [user for user in USERS
              if user in RELEVANT and RATED[USER_INDEX[user]].sum() >= 5]

# TODO: Write metrics(ranked, relevant) returning precision@3, recall@10, NDCG@10 and MAP@10.
# مهمة: اكتب `metrics(ranked, relevant)` تُعيد الدقّة عند ٣ والاستدعاء عند ١٠ وNDCG وMAP.


# TODO: Write evaluate(scorer, label, rows=None) averaging the four metrics over EVAL_USERS.
# مهمة: اكتب `evaluate(scorer, label, rows=None)` تأخذ متوسّط المقاييس الأربعة على `EVAL_USERS`.

MODELS = {"item-based CF": collaborative_scores,
          "content-based": content_scores,
          "popularity baseline": popularity_scores}
RESULTS = pd.DataFrame([evaluate(scorer, label)
                        for label, scorer in MODELS.items()])
RESULTS = RESULTS[["model", "users", "precision@3", "recall@10", "ndcg@10", "map@10"]]

print(f"{len(EVAL_USERS)} evaluable users of {len(USERS)}\n")
print(RESULTS.round(4).to_string(index=False))

best = RESULTS.loc[RESULTS["ndcg@10"].idxmax(), "model"]
print(f"\nbest NDCG@10: {best}")
if best == "popularity baseline":
    print("Recommending the most-rated films to everyone beat both models you built.\n"
          "That is the W1D1 habit arriving on the last taught day: report the baseline,\n"
          "then decide whether the model earned its complexity.")

### Task 2.5 — cold start, both halves

**The new item.** 1,857 films in this catalogue are rated only *after* the split point. The
collaborative recommender cannot recommend a single one of them — they are not in its matrix at all,
so there is no row to score. The content recommender can, because a title and a genre list exist
before anyone has an opinion. Show both: count how many new films each model can rank.

**The thin user.** MovieLens removed everyone with fewer than 20 ratings before publishing, so this
half has to be constructed: take the evaluable users, keep **two** of their training ratings, and
evaluate again. Say in the output that it was constructed — an audience that thinks you found these
users in the data will draw a stronger conclusion than your evidence supports.

**The hybrid.** Content-based when the history is thin, collaborative when it is not. Measure it on
both segments. The interesting result is not that it beats everything; it is what it does on each
segment separately.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — البداية الباردة بنصفيها

**العنصر الجديد.** ألف وثمانمئة وسبعة وخمسون فيلمًا في هذا الفهرس لا تُقيَّم إلا *بعد* نقطة التقسيم.
ولا يستطيع النظام التعاوني ترشيح واحد منها — فليست في مصفوفته أصلًا ولا صفّ لها يُسجَّل. ويستطيع
النظام القائم على المحتوى، لأن العنوان وقائمة الأنواع موجودان قبل أن يكون لأحد رأي. فأظهر الاثنين:
عُدّ كم فيلمًا جديدًا يستطيع كل نموذج ترتيبه.

**والمستخدم الضئيل.** حذفت MovieLens كل من له أقلّ من عشرين تقييمًا قبل النشر، فهذا النصف يجب أن
يُصنع: خذ المستخدمين القابلين للتقييم، وأبقِ **تقييمين** من تدريبهم، وقيّم ثانيةً. وقل في المخرجات
إنه مصنوع — فالجمهور الذي يظنّ أنك وجدت هؤلاء في البيانات سيستنتج أقوى ممّا يحتمله دليلك.

**والهجين.** قائم على المحتوى حين يكون السجلّ ضئيلًا، وتعاوني حين لا يكون. قِسه على القطاعين.
والنتيجة المثيرة ليست أنه يتفوّق على الجميع، بل ما يفعله في كل قطاع على حدة.

</div>

In [ ]:

THIN_HISTORY = 2          # ratings kept per user in the constructed thin segment
THIN_THRESHOLD = 5        # below this many ratings, the hybrid switches to content

# TODO: Count the films that appear only after the split, and how many of them each model could rank at all.
# مهمة: عُدّ الأفلام التي لا تظهر إلا بعد التقسيم، وكم منها يستطيع كل نموذج ترتيبه أصلًا.

print(f"films rated only after the split: {len(NEW_ITEMS):,}")
print(f"  collaborative filtering can rank: {CF_CAN_RANK:,} — it has no column for them")
print(f"  content-based can rank:           {CONTENT_CAN_RANK:,} — a title is enough")

# TODO: Build the constructed thin-history rows, then evaluate CF, content and the hybrid on both the full segment and the thin one.
# مهمة: ابنِ صفوف السجلّ الضئيل المصنوعة، ثم قيّم التعاوني والمحتوى والهجين على القطاع الكامل والقطاع الضئيل.

print()
print(SEGMENTS[["segment", "model", "precision@3", "ndcg@10", "map@10"]]
      .round(4).to_string(index=False))
print(f"\nThe thin segment is constructed: {THIN_HISTORY} ratings kept per user, seeded. "
      f"No real MovieLens user has fewer than 20.")

### Task 2.6 — which metric goes on the phone screen?

Take the warm-up's three orderings again and answer the question they were built for.

A phone shows **three** recommendations. precision@3 measures exactly that surface and is blind to
everything below it — including the difference between "the fourth was relevant" and "nothing else
was relevant at all". NDCG sees the whole list, and on a screen showing three, most of what it sees
is not on the screen.

So the answer is not "the one that changes most". Write the sentence: **which do you report, for
this interface, and what does the other one still tell you?**

<div dir="rtl" align="right">

### المهمة ٢٫٦ — أي مقياس يوضع على شاشة الهاتف؟

خذ ترتيبات الإحماء الثلاثة ثانيةً وأجب عن السؤال الذي بُنيت له.

تعرض الشاشة **ثلاث** توصيات. والدقّة عند ٣ تقيس ذلك السطح بالضبط وتعمى عمّا دونه — ومنه الفرق بين
«كان الرابع ذا صلة» و«لم يكن شيء آخر ذا صلة أصلًا». وNDCG يرى القائمة كلها، وعلى شاشةٍ تعرض ثلاثة
يكون أكثر ما يراه خارج الشاشة.

فالجواب ليس «الذي يتغيّر أكثر». اكتب الجملة: **أيّهما تذكر لهذه الواجهة، وماذا يظلّ الآخر يخبرك به؟**

</div>

In [ ]:

# TODO: Build the three-row comparison table with precision@3, NDCG and the change from the first ordering.
# مهمة: ابنِ جدول المقارنة ذا الصفوف الثلاثة بالدقّة عند ٣ وNDCG والفرق عن الترتيب الأول.

print(METRIC_TABLE.round(4).to_string(index=False))
PRECISION_BLIND = np.isclose(WARM_METRICS["1, 2, 5"][0], WARM_METRICS["1, 2, 4"][0])
NDCG_SEES = WARM_METRICS["1, 2, 4"][1] > WARM_METRICS["1, 2, 5"][1]
print(f"\nmoving position 5 → 4: precision@3 unchanged: {PRECISION_BLIND}, "
      f"NDCG rose: {NDCG_SEES}")

**Your sentence.** Replace this text.

> For a screen showing three items I would report … , because … . The other metric still tells me
> … .

<div dir="rtl" align="right">

**جملتك.** استبدل هذا النصّ.

> لشاشة تعرض ثلاثة عناصر أذكر … لأن … . ويظلّ المقياس الآخر يخبرني بـ … .

</div>

## Section 3 — Stretch: search, and the rest of applied NLP  (≈30 min)

**(a) Semantic search, closing the loop.** W6D5 searched 200 images by scanning every vector.
W7D2 built an index over `policy_docs`. Write a five-line search function over the Chroma store you
persisted on Tuesday and time it against a brute-force scan of the same vectors. On this corpus
brute force may well win — 85 vectors is nothing — and the interesting output is the size at which
that stops being true.

**(b) The applied-NLP survey.** Four `pipeline` calls on one policy document: named-entity
recognition, zero-shot classification, sentiment (in both languages), and text generation. One
sentence each on where you would use it, and **one sentence on how it failed** — every one of them
fails visibly on this text, and finding the failure is the exercise.

**A note on the API.** `transformers` 5 removed the `summarization`, `translation` and
`question-answering` pipeline shortcuts; those models are still there and are now loaded through
`AutoModelForSeq2SeqLM` and called with `.generate()`. Library churn of exactly this kind is why
your capstone pins its versions.

<div dir="rtl" align="right">

## القسم الثالث — التوسّع: البحث وبقية معالجة اللغة التطبيقية (نحو ٣٠ دقيقة)

**(أ) البحث الدلالي، وإغلاق الحلقة.** بحث الأسبوع السادس اليوم الخامس في مئتَي صورة بمسح كل متّجه.
وبنى الأسبوع السابع اليوم الثاني فهرسًا على `policy_docs`. فاكتب دالة بحث من خمسة أسطر على مخزن
Chroma الذي حفظته الثلاثاء، وقِس زمنها مقابل مسح شامل للمتّجهات نفسها. وقد يفوز المسح الشامل في هذه
المُدوّنة — فخمسة وثمانون متّجهًا لا شيء — والمخرَج المثير هو الحجم الذي يبطل عنده ذلك.

**(ب) جولة معالجة اللغة التطبيقية.** أربعة نداءات `pipeline` على وثيقة سياسات واحدة: التعرّف على
الكيانات المسمّاة، والتصنيف بلا تدريب، وتحليل المشاعر باللغتين، وتوليد النصّ. جملة لكل واحد عن أين
تستعمله، و**جملة عن كيف أخفق** — فكلها تُخفق ظاهرًا على هذا النصّ، وإيجاد الإخفاق هو التمرين.

**وملاحظة عن الواجهة:** حذف الإصدار الخامس من `transformers` اختصارات `summarization` و
`translation` و`question-answering`، ونماذجها ما تزال موجودة وتُحمَّل الآن بـ`AutoModelForSeq2SeqLM`
وتُستدعى بـ`.generate()`. وتقلّب المكتبات من هذا النوع بعينه هو سبب تثبيت مشروعك لإصداراته.

</div>

In [ ]:

import time
from aiep.data import load_artefact

# TODO: Open Tuesday's Chroma store, run one query through it, and time it against a brute-force scan of the same vectors.
# مهمة: افتح مخزن Chroma من الثلاثاء، وشغّل استعلامًا واحدًا فيه، وقِس زمنه مقابل مسح شامل للمتّجهات نفسها.

print(f'query: "{QUERY}"')
print(f"  chroma  {INDEX_MS:.2f} ms → {indexed_hit['ids'][0]}")
print(f"  brute   {BRUTE_MS:.2f} ms → {[stored['ids'][i] for i in brute_order]}")
print(f"\n{len(STORE_VECTORS)} vectors. Brute force is {'faster' if BRUTE_MS < INDEX_MS else 'slower'} "
      f"here, and that is the honest result at this size — the index earns its keep when the scan "
      f"stops fitting in cache, which is four or five orders of magnitude from here.")

In [ ]:

# Four tools on one document. Two of these models you have already downloaded in week 5,
# and the whole cell is about a hundred megabytes if you have not.
from transformers import pipeline

DOCUMENT = (
    "Olo Retail accepts returns on most items sold through its online store and its branches "
    "in Riyadh, Jeddah and Dammam. A refund is issued to the original payment method within 14 "
    "days of purchase. Digital goods are excluded from that window. Expenses are claimed on form "
    "27B in the staff portal, and a claim submitted more than 60 days after the expense is not "
    "paid at all.")

entities = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
print("NER:", [(e["word"], e["entity_group"]) for e in entities(DOCUMENT)])

labeller = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli")
labelled = labeller(DOCUMENT, candidate_labels=["returns and refunds", "staff expenses",
                                                "information security", "delivery"])
print("\nZero-shot:", [(label, round(score, 3))
                       for label, score in zip(labelled["labels"], labelled["scores"])])

sentiment = pipeline("sentiment-analysis",
                     model="nlptown/bert-base-multilingual-uncased-sentiment")
for text in [DOCUMENT[:120], "خدمة سيئة جدًّا ولم يُرجع لي المبلغ"]:
    result = sentiment(text)[0]
    print(f"\nSentiment: {result['label']} ({result['score']:.2f}) ← {text[:60]}…")

writer = pipeline("text-generation", model="distilgpt2")
prompt = "Olo Retail's refund window for digital goods is"
print("\nGeneration:", writer(prompt, max_new_tokens=30, do_sample=False,
                              truncation=True)[0]["generated_text"])
print("\nThat last line is Wednesday's lesson one more time: the generator will finish the "
      "sentence whether or not it knows the answer.")

**Four sentences, one per tool.** Where would you use it, and how did it fail here?

> NER: …
>
> Zero-shot classification: …
>
> Sentiment: …
>
> Text generation: …

<div dir="rtl" align="right">

**أربع جمل، واحدة لكل أداة.** أين تستعملها، وكيف أخفقت هنا؟

> التعرّف على الكيانات: …
>
> التصنيف بلا تدريب: …
>
> تحليل المشاعر: …
>
> توليد النصّ: …

</div>

## Save the artefact

`recsys_results.parquet` holds every model on every segment with all four metrics, and the
cold-start counts go with it. It is the table your capstone report would carry if your project is a
recommender — with the baseline row still in it.

<div dir="rtl" align="right">

## احفظ المُخرَج

يحوي `recsys_results.parquet` كل نموذج في كل قطاع بالمقاييس الأربعة، وتذهب معه أعداد البداية
الباردة. وهو الجدول الذي يحمله تقرير مشروعك إن كان مشروعك نظام توصية — وفيه صفّ خطّ الأساس.

</div>

In [ ]:
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

COLD_START = pd.DataFrame([
    {"segment": "brand-new items", "model": "item-based CF", "rankable": CF_CAN_RANK,
     "candidates": len(NEW_ITEMS)},
    {"segment": "brand-new items", "model": "content-based", "rankable": CONTENT_CAN_RANK,
     "candidates": len(NEW_ITEMS)},
])
SEGMENTS.to_parquet(ARTEFACT_DIR / "recsys_results.parquet", index=False)
COLD_START.to_parquet(ARTEFACT_DIR / "cold_start.parquet", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
for segment, group in SEGMENTS.groupby("segment"):
    ax.bar([f"{m}\n{segment.split()[0]}" for m in group.model], group["ndcg@10"], label=segment)
ax.set_ylabel("NDCG@10")
ax.set_title("every model, both segments, with the baseline in the picture")
ax.legend(fontsize=8)
plt.xticks(fontsize=7)
savefig(fig, "recsys_ndcg.png")
plt.show()

print(SEGMENTS[["segment", "model", "precision@3", "recall@10", "ndcg@10", "map@10"]]
      .round(4).to_string(index=False))

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check_close(WARM_SIM_AD, 0.9939,
            "sim(A, D) must be 0.9939 as on slide 25",
            "يجب أن يكون sim(A, D) مساويًا 0.9939 كما في الشريحة ٢٥",
            tol=0.0005)

check(WARM_SIM_AC is None and WARM_CO_BD == 1 and np.isclose(WARM_SIM_BD, 1.0),
      f"sim(A, C) must be undefined rather than zero (got {WARM_SIM_AC}), and sim(B, D) must be "
      f"exactly 1.0000 from exactly one co-rater (got {WARM_SIM_BD:.4f} from {WARM_CO_BD}). Both "
      f"traps are the reason this warm-up exists",
      f"يجب أن يكون sim(A, C) غير معرَّف لا صفرًا (والناتج {WARM_SIM_AC})، وأن يكون sim(B, D) مساويًا "
      f"1.0000 تمامًا من مقيّم مشترك واحد (والناتج {WARM_SIM_BD:.4f} من {WARM_CO_BD}). والفخّان هما "
      f"سبب وجود هذا الإحماء")

check_close(PREDICTION, 3.50,
            "the predicted rating for U2 on D must be 3.50 to two decimals",
            "يجب أن يكون التقييم المتنبّأ به لـU2 على D مساويًا 3.50 إلى منزلتين",
            tol=0.005)

check(bool(PRECISION_BLIND and NDCG_SEES)
      and np.isclose(WARM_METRICS["1, 4, 5"][0], 1 / 3, atol=0.001)
      and np.isclose(WARM_METRICS["1, 4, 5"][1], 0.8529, atol=0.001),
      f"the ranking numbers must reproduce the slide: 0.667/0.9469, 0.333/0.8529, 0.667/0.9675 — "
      f"and precision@3 must stay still while NDCG rises on the 5→4 move (still: {PRECISION_BLIND}, "
      f"rose: {NDCG_SEES}). That contrast is the argument for NDCG",
      f"يجب أن تُعاد أرقام الترتيب كما في الشريحة: 0.667/0.9469 و0.333/0.8529 و0.667/0.9675 — وأن "
      f"تثبت الدقّة عند ٣ بينما يرتفع NDCG عند النقل من ٥ إلى ٤ (ثبتت: {PRECISION_BLIND}، ارتفع: "
      f"{NDCG_SEES}). وهذا التباين هو حجّة NDCG")

check(0.95 < FULL_SPARSITY < 0.995,
      f"the full rating matrix must be about 98% empty — got {FULL_SPARSITY:.1%}. Every method "
      f"today is an attempt to speak about that emptiness from what is left",
      f"يجب أن تكون مصفوفة التقييمات الكاملة فارغة بنحو ٩٨٪ — والناتج {FULL_SPARSITY:.1%}. وكل طريقة "
      f"اليوم محاولةٌ للحديث عن ذلك الفراغ ممّا تبقّى")

check(bool(np.isnan(SIMILARITY[CO_RATERS == 0]).all()) and LONELY_PAIRS > 0,
      f"no similarity may be computed from zero co-raters — every such pair must be NaN, not 0.0 — "
      f"and the count resting on a single co-rater must be recorded ({LONELY_PAIRS:,} pairs). A "
      f"zero there is a claim that two films are unrelated, which is not what the data said",
      f"لا يجوز حساب تشابه من صفر مقيّمين مشتركين — فكل زوج كهذا `NaN` لا `0.0` — ويجب تسجيل عدد ما "
      f"يقوم على مقيّم واحد ({LONELY_PAIRS:,} زوجًا). فالصفر هناك ادّعاءٌ بأن الفيلمين لا صلة بينهما، "
      f"وليس هذا ما قالته البيانات")

TRAIN_MAX = TRAIN.groupby("userId").timestamp.max()
TEST_MIN = TEST.groupby("userId").timestamp.min()
shared_users = TRAIN_MAX.index.intersection(TEST_MIN.index)
CHRONOLOGICAL = bool((TRAIN_MAX[shared_users] <= TEST_MIN[shared_users]).all())
check(CHRONOLOGICAL,
      f"the split must be chronological for every user — each user's last training rating must "
      f"come before their first test rating (holds for all {len(shared_users)} users: "
      f"{CHRONOLOGICAL}). A random split trains on the future and scores on the past",
      f"يجب أن يكون التقسيم زمنيًّا لكل مستخدم — فآخر تقييم تدريب له قبل أول تقييم اختبار "
      f"(يتحقّق لكل المستخدمين البالغ عددهم {len(shared_users)}: {CHRONOLOGICAL}). والتقسيم العشوائي "
      f"يدرّب على المستقبل ويقيس على الماضي")

check(len(RESULTS) == 3 and RESULTS[["precision@3", "recall@10", "ndcg@10", "map@10"]].notna().all().all()
      and "popularity baseline" in set(RESULTS.model),
      f"all four metrics must be present for all three models, and the popularity baseline must be "
      f"one of them — models present: {sorted(RESULTS.model)}. A recommender reported without a "
      f"baseline beside it is a number with nothing to compare to",
      f"يجب أن تكون المقاييس الأربعة موجودة للنماذج الثلاثة، وأن يكون خطّ الأساس الشعبي أحدها — "
      f"والموجود: {sorted(RESULTS.model)}. ونظام التوصية المذكور بلا خطّ أساس بجواره رقمٌ بلا ما يُقارن به")

check(CF_CAN_RANK == 0 and CONTENT_CAN_RANK == len(NEW_ITEMS),
      f"collaborative filtering must be unable to rank any of the {len(NEW_ITEMS):,} brand-new "
      f"films (it ranked {CF_CAN_RANK}) while the content model ranks all of them "
      f"({CONTENT_CAN_RANK}). That gap is cold start, demonstrated rather than described",
      f"يجب أن يعجز الترشيح التعاوني عن ترتيب أي فيلم من الأفلام الجديدة البالغة {len(NEW_ITEMS):,} "
      f"(ورتّب {CF_CAN_RANK}) بينما يرتّبها النموذج القائم على المحتوى كلها ({CONTENT_CAN_RANK}). "
      f"وتلك الفجوة هي البداية الباردة معروضةً لا موصوفة")

report()

## What's next

**Week 8 — your project, and getting it out of the notebook.** Monday is the last authored lab of
the course: model artifacts and serving, where you reproduce three ways a working model fails to
ship and then fix all three on your own capstone. Tuesday to Thursday are clinics, a report
workshop and a timed dress rehearsal. Friday you demo.

Two things from today go with you. The **baseline** — no result gets reported without one beside
it, and on demo day the first question is "what is your baseline and what does it score". And the
**metric you chose on purpose**, which is the second question in every interview that follows this
course.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع الثامن — مشروعك، وإخراجه من الدفتر.** الاثنين آخر معمل مؤلَّف في المقرّر: مُخرَجات النماذج
وتقديمها، حيث تُعيد إنتاج ثلاث طرق يُخفق بها نموذج عامل في الوصول إلى الإنتاج ثم تصلح الثلاث في
مشروعك أنت. ومن الثلاثاء إلى الخميس عيادات وورشة تقرير وبروفة مؤقّتة. والجمعة تعرض.

ويرافقك شيئان من اليوم. **خطّ الأساس** — فلا نتيجة تُذكر بلا خطّ أساس بجوارها، وأول سؤال يوم العرض
«ما خطّ أساسك وكم يسجّل؟». و**المقياس الذي اخترته عن قصد**، وهو ثاني سؤال في كل مقابلة بعد هذا
المقرّر.

</div>